# Description

In this notebook, I will implement the multi-head attention

In [12]:
import os 
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

In [10]:
BATCH_SIZE = 1
SEQUENCE_LENGTH = 8
D_MODEL = 128

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float16

Q = torch.randn(SEQUENCE_LENGTH, BATCH_SIZE, D_MODEL, device=device, dtype=dtype)
K = torch.randn(SEQUENCE_LENGTH, BATCH_SIZE, D_MODEL, device=device, dtype=dtype)
V = torch.randn(SEQUENCE_LENGTH, BATCH_SIZE, D_MODEL, device=device, dtype=dtype)

In [19]:
# Torch built-in multi-head attention
mha = torch.nn.MultiheadAttention(embed_dim=D_MODEL, num_heads=8, device=device, dtype=dtype)
output, _ = mha(Q, K, V)
print("Output shape:", output.shape)

Output shape: torch.Size([8, 1, 128])


In [20]:
import math
from typing import Optional

import torch
import torch.nn as nn
import torch.nn.functional as F


class MultiHeadAttentionScratch(nn.Module):
    """
    Multi-Head Self/Encoder-Decoder Attention implemented from scratch using only
    matmul + reshapes (no nn.Linear layers).

    Shapes use batch-first convention: (B, T, D).

    Args:
        d_model: model (embedding) dimension.
        num_heads: number of attention heads.
        dropout_p: dropout probability on attention weights.
        bias: whether to include bias terms for the projections.
        causal: if True, applies a causal mask (no peeking ahead).
    """

    def __init__(
        self,
        d_model: int,
        num_heads: int,
        dropout_p: float = 0.0
    ):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads
        self.dropout_p = dropout_p

        # Projection weights (no nn.Linear) — just Parameters used with matmul
        # Input: (B, T, D) -> (B, T, D) per projection
        self.W_q = nn.Parameter(torch.empty(d_model, d_model))
        self.W_k = nn.Parameter(torch.empty(d_model, d_model))
        self.W_v = nn.Parameter(torch.empty(d_model, d_model))

        # Output projection: concat heads (B, T, H*Dh=D) -> (B, T, D)
        self.W_o = nn.Parameter(torch.empty(d_model, d_model))

        self.reset_parameters()

    def reset_parameters(self):
        # Xavier init similar to nn.Linear defaults
        for W in (self.W_q, self.W_k, self.W_v, self.W_o):
            nn.init.xavier_uniform_(W)

    @staticmethod
    def _split_heads(x: torch.Tensor, num_heads: int) -> torch.Tensor:
        """
        (B, T, D) -> (B, H, T, Dh)
        """
        B, T, D = x.shape
        Dh = D // num_heads
        x = x.view(B, T, num_heads, Dh)
        return x.permute(0, 2, 1, 3)

    @staticmethod
    def _merge_heads(x: torch.Tensor) -> torch.Tensor:
        """
        (B, H, T, Dh) -> (B, T, H*Dh)
        """
        B, H, T, Dh = x.shape
        x = x.permute(0, 2, 1, 3).contiguous()
        return x.view(B, T, H * Dh)

    def _project(self, x: torch.Tensor, W: torch.Tensor, b: Optional[torch.Tensor]) -> torch.Tensor:
        """
        Affine projection using matmul only.
        x: (B, T, D), W: (D, D), b: (D,) or None -> (B, T, D)
        """
        # (B, T, D) @ (D, D) -> (B, T, D)
        y = torch.matmul(x, W)
        return y

    def forward(
        self,
        x_q: torch.Tensor,
        x_kv: Optional[torch.Tensor] = None,
        key_padding_mask: Optional[torch.Tensor] = None,
    ):
        """
        Args:
            x_q: query input (B, T_q, D)
            x_kv: key/value input (B, T_kv, D). If None, uses x_q (self-attention).
            key_padding_mask: boolean mask (B, T_kv), True for positions to mask (padding).
        Returns:
            y: output (B, T_q, D)
            attn_probs_avg (optional): (B, T_q, T_kv)
        """
        if x_kv is None:
            x_kv = x_q

        B, T_q, D = x_q.shape
        B2, T_kv, D2 = x_kv.shape
        assert D == self.d_model and D2 == self.d_model and B == B2

        # Projections via matmul
        Q = torch.matmul(x_q, self.W_q)  # (B, T_q, D)
        K = torch.matmul(x_kv, self.W_k)  # (B, T_kv, D)
        V = torch.matmul(x_kv, self.W_v)  # (B, T_kv, D)

        # Split into heads
        Q = self._split_heads(Q, self.num_heads)  # (B, H, T_q, Dh)
        K = self._split_heads(K, self.num_heads)  # (B, H, T_kv, Dh)
        V = self._split_heads(V, self.num_heads)  # (B, H, T_kv, Dh)

        # Scaled dot-product attention: (B, H, T_q, Dh) @ (B, H, Dh, T_kv) -> (B, H, T_q, T_kv)
        scale = 1.0 / math.sqrt(self.d_head)
        scores = torch.matmul(Q, K.transpose(-1, -2)) * scale

        attn_probs = F.softmax(scores, dim=-1)

        # (B, H, T_q, T_kv) @ (B, H, T_kv, Dh) -> (B, H, T_q, Dh)
        context = torch.matmul(attn_probs, V)

        # Merge heads and output projection
        context = self._merge_heads(context)  # (B, T_q, D)
        y = torch.matmul(context, self.W_o)  # (B, T_q, D)

        return y

In [21]:
# Test the custom MultiHeadAttentionScratch
mha_scratch = MultiHeadAttentionScratch(d_model=D_MODEL, num_heads=8).to(device=device, dtype=dtype)
output_scratch = mha_scratch(Q.permute(1, 0, 2))  # (B, T, D)
print("Custom Output shape:", output_scratch.shape)

Custom Output shape: torch.Size([1, 8, 128])
